# 🐍 Snake Game v1 - 기본 버전

18×18 그리드의 기본 스네이크 게임입니다.

**기능:**
- 18×18 격자판 (셀 크기 28px)
- 방향키로 뱀 조작
- 먹이 섭취 시 +10점
- 50점마다 레벨 업 (속도 증가)
- 점수 / 레벨 / 최고점수(localStorage) HUD
- 벽 또는 자기 몸 충돌 시 게임 오버
- R키로 재시작

## 📦 CSS 스타일 정의

In [ ]:
css = """
/* 전체 페이지 기본 스타일 */
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

/* 바디 배경 및 레이아웃 */
body {
    background-color: #1a1a2e;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 100vh;
    font-family: 'Courier New', Courier, monospace;
    color: #e0e0e0;
}

/* 게임 컨테이너 */
.game-container {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 12px;
    padding: 16px;
}

/* 게임 제목 */
.game-title {
    font-size: 28px;
    font-weight: bold;
    color: #4ade80;
    letter-spacing: 4px;
    text-transform: uppercase;
    text-shadow: 0 0 10px rgba(74, 222, 128, 0.5);
}

/* HUD (점수판) 스타일 */
.hud {
    display: flex;
    gap: 24px;
    background: rgba(255, 255, 255, 0.05);
    border: 1px solid rgba(74, 222, 128, 0.3);
    border-radius: 8px;
    padding: 8px 20px;
}

/* HUD 개별 항목 */
.hud-item {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 2px;
}

/* HUD 레이블 */
.hud-label {
    font-size: 10px;
    color: #9ca3af;
    letter-spacing: 2px;
    text-transform: uppercase;
}

/* HUD 값 */
.hud-value {
    font-size: 22px;
    font-weight: bold;
    color: #4ade80;
    font-variant-numeric: tabular-nums;
}

/* 캔버스 스타일 */
canvas {
    border: 2px solid rgba(74, 222, 128, 0.4);
    border-radius: 4px;
    display: block;
    box-shadow: 0 0 20px rgba(74, 222, 128, 0.15);
}

/* 조작 안내 텍스트 */
.instructions {
    font-size: 11px;
    color: #6b7280;
    letter-spacing: 1px;
    text-align: center;
}
"""

## 🏗️ HTML 구조

In [ ]:
html_body = """
<div class="game-container">
    <!-- 게임 제목 -->
    <div class="game-title">🐍 Snake</div>

    <!-- 점수 HUD -->
    <div class="hud">
        <div class="hud-item">
            <span class="hud-label">Score</span>
            <span class="hud-value" id="score">0</span>
        </div>
        <div class="hud-item">
            <span class="hud-label">Level</span>
            <span class="hud-value" id="level">1</span>
        </div>
        <div class="hud-item">
            <span class="hud-label">Best</span>
            <span class="hud-value" id="best">0</span>
        </div>
    </div>

    <!-- 게임 캔버스 -->
    <canvas id="gameCanvas" width="504" height="504"></canvas>

    <!-- 조작 안내 -->
    <div class="instructions">방향키로 이동 &nbsp;|&nbsp; R 키로 재시작</div>
</div>
"""

## 🎮 JavaScript 게임 로직

In [ ]:
js = """
// 캔버스와 컨텍스트 가져오기
const canvas = document.getElementById('gameCanvas');
const ctx = canvas.getContext('2d');

// 게임 상수 정의
const GRID_SIZE = 18;          // 그리드 크기 (18x18)
const CELL_SIZE = 28;          // 각 셀의 픽셀 크기
const BASE_SPEED = 250;        // 초기 이동 속도 (밀리초)
const SPEED_DECREASE = 15;     // 레벨당 속도 감소량
const MIN_SPEED = 80;          // 최소 이동 속도 (최대 빠르기)
const POINTS_PER_FOOD = 10;    // 먹이 하나당 획득 점수
const POINTS_PER_LEVEL = 50;   // 레벨 업에 필요한 점수

// 색상 상수 정의
const COLOR_BG = '#1a1a2e';          // 배경색 (짙은 남색)
const COLOR_GRID = '#16213e';        // 그리드 선 색상
const COLOR_SNAKE_HEAD = '#22c55e';  // 뱀 머리 색상 (밝은 초록)
const COLOR_SNAKE_BODY = '#4ade80';  // 뱀 몸통 색상 (연한 초록)
const COLOR_FOOD = '#ef4444';        // 먹이 색상 (빨강)
const COLOR_FOOD_GLOW = '#fca5a5';   // 먹이 빛나는 효과 색상
const COLOR_GAMEOVER = 'rgba(0,0,0,0.75)'; // 게임오버 오버레이 색상

// 게임 상태 변수 선언
let snake = [];        // 뱀의 위치 배열 (머리가 인덱스 0)
let food = {};         // 먹이의 위치 {x, y}
let direction = {};    // 현재 이동 방향 {x, y}
let nextDirection = {}; // 다음 프레임 이동 방향
let score = 0;         // 현재 점수
let level = 1;         // 현재 레벨
let bestScore = 0;     // 최고 점수
let gameOver = false;  // 게임 오버 여부
let gameLoop = null;   // 게임 루프 타이머 ID
let gameStarted = false; // 게임 시작 여부

// localStorage에서 최고 점수 불러오기
function loadBestScore() {
    try {
        // 저장된 최고 점수 가져오기
        const saved = localStorage.getItem('snakeV1Best');
        bestScore = saved ? parseInt(saved, 10) : 0;
    } catch (e) {
        // localStorage 접근 실패 시 0으로 초기화
        bestScore = 0;
    }
    // HUD에 최고 점수 표시
    document.getElementById('best').textContent = bestScore;
}

// localStorage에 최고 점수 저장하기
function saveBestScore() {
    try {
        // 현재 점수가 최고 점수보다 높으면 저장
        if (score > bestScore) {
            bestScore = score;
            localStorage.setItem('snakeV1Best', bestScore);
            document.getElementById('best').textContent = bestScore;
        }
    } catch (e) {
        // localStorage 저장 실패 시 무시
    }
}

// 게임 초기화 함수
function initGame() {
    // 기존 게임 루프 정지
    if (gameLoop) clearInterval(gameLoop);

    // 뱀을 그리드 중앙에 배치 (길이 3)
    const center = Math.floor(GRID_SIZE / 2);
    snake = [
        { x: center,     y: center },      // 머리
        { x: center - 1, y: center },      // 몸통 1
        { x: center - 2, y: center }       // 몸통 2 (꼬리)
    ];

    // 초기 이동 방향: 오른쪽
    direction = { x: 1, y: 0 };
    nextDirection = { x: 1, y: 0 };

    // 점수와 레벨 초기화
    score = 0;
    level = 1;
    gameOver = false;
    gameStarted = true;

    // HUD 업데이트
    document.getElementById('score').textContent = 0;
    document.getElementById('level').textContent = 1;

    // 먹이 생성
    spawnFood();

    // 게임 루프 시작
    gameLoop = setInterval(update, BASE_SPEED);
}

// 뱀이 없는 위치에 먹이 생성
function spawnFood() {
    let pos;
    do {
        // 랜덤 위치 생성
        pos = {
            x: Math.floor(Math.random() * GRID_SIZE),
            y: Math.floor(Math.random() * GRID_SIZE)
        };
        // 뱀 위치와 겹치지 않을 때까지 반복
    } while (snake.some(seg => seg.x === pos.x && seg.y === pos.y));
    food = pos;
}

// 현재 레벨에 맞는 이동 속도 계산
function getSpeed() {
    // 레벨이 높을수록 속도 증가 (간격 감소)
    const speed = BASE_SPEED - (level - 1) * SPEED_DECREASE;
    return Math.max(speed, MIN_SPEED);
}

// 게임 업데이트 (매 프레임)
function update() {
    // 방향 업데이트
    direction = { ...nextDirection };

    // 새 머리 위치 계산
    const head = {
        x: snake[0].x + direction.x,
        y: snake[0].y + direction.y
    };

    // 벽 충돌 확인
    if (head.x < 0 || head.x >= GRID_SIZE || head.y < 0 || head.y >= GRID_SIZE) {
        endGame(); // 벽에 닿으면 게임 오버
        return;
    }

    // 자기 몸통 충돌 확인
    if (snake.some(seg => seg.x === head.x && seg.y === head.y)) {
        endGame(); // 자기 몸에 닿으면 게임 오버
        return;
    }

    // 뱀 앞으로 이동 (머리에 새 위치 추가)
    snake.unshift(head);

    // 먹이 섭취 확인
    if (head.x === food.x && head.y === food.y) {
        // 점수 증가
        score += POINTS_PER_FOOD;
        document.getElementById('score').textContent = score;

        // 레벨 업 확인
        const newLevel = Math.floor(score / POINTS_PER_LEVEL) + 1;
        if (newLevel > level) {
            level = newLevel;
            document.getElementById('level').textContent = level;
            // 속도 업데이트
            clearInterval(gameLoop);
            gameLoop = setInterval(update, getSpeed());
        }

        // 새 먹이 생성 (꼬리 유지)
        spawnFood();
    } else {
        // 먹이 없으면 꼬리 제거 (이동)
        snake.pop();
    }

    // 화면 다시 그리기
    draw();
}

// 게임 오버 처리
function endGame() {
    // 게임 루프 정지
    clearInterval(gameLoop);
    gameOver = true;

    // 최고 점수 저장
    saveBestScore();

    // 게임 오버 화면 그리기
    draw();
    drawGameOver();
}

// 게임 오버 오버레이 그리기
function drawGameOver() {
    // 반투명 오버레이
    ctx.fillStyle = COLOR_GAMEOVER;
    ctx.fillRect(0, 0, canvas.width, canvas.height);

    // 게임 오버 텍스트
    ctx.fillStyle = '#ef4444';
    ctx.font = 'bold 36px Courier New';
    ctx.textAlign = 'center';
    ctx.fillText('GAME OVER', canvas.width / 2, canvas.height / 2 - 30);

    // 점수 표시
    ctx.fillStyle = '#e0e0e0';
    ctx.font = '20px Courier New';
    ctx.fillText('점수: ' + score, canvas.width / 2, canvas.height / 2 + 10);

    // 재시작 안내
    ctx.fillStyle = '#4ade80';
    ctx.font = '16px Courier New';
    ctx.fillText('R 키를 눌러 재시작', canvas.width / 2, canvas.height / 2 + 45);
    ctx.textAlign = 'left';
}

// 그리드 그리기
function drawGrid() {
    ctx.strokeStyle = COLOR_GRID;
    ctx.lineWidth = 0.5;

    // 세로 선 그리기
    for (let x = 0; x <= GRID_SIZE; x++) {
        ctx.beginPath();
        ctx.moveTo(x * CELL_SIZE, 0);
        ctx.lineTo(x * CELL_SIZE, canvas.height);
        ctx.stroke();
    }

    // 가로 선 그리기
    for (let y = 0; y <= GRID_SIZE; y++) {
        ctx.beginPath();
        ctx.moveTo(0, y * CELL_SIZE);
        ctx.lineTo(canvas.width, y * CELL_SIZE);
        ctx.stroke();
    }
}

// 먹이 그리기
function drawFood() {
    // 먹이 중심 좌표 계산
    const cx = food.x * CELL_SIZE + CELL_SIZE / 2;
    const cy = food.y * CELL_SIZE + CELL_SIZE / 2;
    const radius = CELL_SIZE / 2 - 3;

    // 빛나는 효과 그리기
    ctx.shadowBlur = 12;
    ctx.shadowColor = COLOR_FOOD_GLOW;

    // 먹이 원 그리기
    ctx.fillStyle = COLOR_FOOD;
    ctx.beginPath();
    ctx.arc(cx, cy, radius, 0, Math.PI * 2);
    ctx.fill();

    // 광택 효과 (위쪽 흰색 하이라이트)
    ctx.fillStyle = 'rgba(255, 255, 255, 0.4)';
    ctx.beginPath();
    ctx.arc(cx - 2, cy - 3, radius * 0.35, 0, Math.PI * 2);
    ctx.fill();

    // 그림자 효과 초기화
    ctx.shadowBlur = 0;
}

// 뱀 그리기
function drawSnake() {
    // 각 세그먼트 그리기
    snake.forEach((seg, index) => {
        // 머리는 밝은 색, 몸통은 연한 색
        ctx.fillStyle = index === 0 ? COLOR_SNAKE_HEAD : COLOR_SNAKE_BODY;

        // 세그먼트 위치 계산 (약간 패딩 추가)
        const x = seg.x * CELL_SIZE + 1;
        const y = seg.y * CELL_SIZE + 1;
        const size = CELL_SIZE - 2;

        // 사각형으로 세그먼트 그리기
        ctx.fillRect(x, y, size, size);

        // 머리에 눈 그리기
        if (index === 0) {
            ctx.fillStyle = '#1a1a2e';
            // 방향에 따라 눈 위치 조정
            const eyeOffset = 5;
            const cx = seg.x * CELL_SIZE + CELL_SIZE / 2;
            const cy = seg.y * CELL_SIZE + CELL_SIZE / 2;

            // 왼쪽 눈
            ctx.beginPath();
            ctx.arc(
                cx + direction.y * eyeOffset - direction.x * 3,
                cy - direction.x * eyeOffset - direction.y * 3,
                2, 0, Math.PI * 2
            );
            ctx.fill();

            // 오른쪽 눈
            ctx.beginPath();
            ctx.arc(
                cx - direction.y * eyeOffset - direction.x * 3,
                cy + direction.x * eyeOffset - direction.y * 3,
                2, 0, Math.PI * 2
            );
            ctx.fill();
        }
    });
}

// 전체 화면 그리기
function draw() {
    // 배경 지우기
    ctx.fillStyle = COLOR_BG;
    ctx.fillRect(0, 0, canvas.width, canvas.height);

    // 그리드 그리기
    drawGrid();

    // 먹이 그리기
    drawFood();

    // 뱀 그리기
    drawSnake();
}

// 시작 화면 그리기
function drawStartScreen() {
    // 배경 그리기
    ctx.fillStyle = COLOR_BG;
    ctx.fillRect(0, 0, canvas.width, canvas.height);

    // 그리드 그리기
    drawGrid();

    // 제목 텍스트
    ctx.fillStyle = '#4ade80';
    ctx.font = 'bold 40px Courier New';
    ctx.textAlign = 'center';
    ctx.fillText('SNAKE', canvas.width / 2, canvas.height / 2 - 40);

    // 시작 안내 텍스트
    ctx.fillStyle = '#e0e0e0';
    ctx.font = '18px Courier New';
    ctx.fillText('방향키로 시작하세요', canvas.width / 2, canvas.height / 2 + 10);

    // 버전 표시
    ctx.fillStyle = '#6b7280';
    ctx.font = '13px Courier New';
    ctx.fillText('v1 - 기본 버전', canvas.width / 2, canvas.height / 2 + 45);
    ctx.textAlign = 'left';
}

// 키보드 이벤트 처리
document.addEventListener('keydown', (e) => {
    // 재시작 키 처리 (게임 오버 상태에서)
    if (e.key === 'r' || e.key === 'R') {
        initGame();
        return;
    }

    // 게임 시작 전이면 첫 키 입력으로 게임 시작
    if (!gameStarted) {
        initGame();
    }

    // 방향 키 처리
    switch (e.key) {
        case 'ArrowUp':
            // 현재 아래로 이동 중이면 위로 전환 불가 (역방향 금지)
            if (direction.y !== 1) nextDirection = { x: 0, y: -1 };
            e.preventDefault(); // 페이지 스크롤 방지
            break;
        case 'ArrowDown':
            // 현재 위로 이동 중이면 아래로 전환 불가
            if (direction.y !== -1) nextDirection = { x: 0, y: 1 };
            e.preventDefault();
            break;
        case 'ArrowLeft':
            // 현재 오른쪽으로 이동 중이면 왼쪽 전환 불가
            if (direction.x !== 1) nextDirection = { x: -1, y: 0 };
            e.preventDefault();
            break;
        case 'ArrowRight':
            // 현재 왼쪽으로 이동 중이면 오른쪽 전환 불가
            if (direction.x !== -1) nextDirection = { x: 1, y: 0 };
            e.preventDefault();
            break;
    }
});

// 게임 초기 상태: 시작 화면 표시
loadBestScore();    // 최고 점수 불러오기
drawStartScreen();  // 시작 화면 그리기
"""

## 🚀 게임 실행

In [ ]:
from IPython.display import IFrame

# HTML, CSS, JS를 하나의 완전한 HTML 파일로 합치기
full_html = f"""<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Snake Game v1</title>
    <style>{css}</style>
</head>
<body>
{html_body}
<script>{js}</script>
</body>
</html>"""

# HTML 파일로 저장
with open('snake_game.html', 'w', encoding='utf-8') as f:
    f.write(full_html)

# IFrame으로 게임 표시
IFrame(src='snake_game.html', width=560, height=620)